# 04 — Modeling Prep
**Source:** `data/silver/` + features from notebook 03

**Goal:** Build a clean, ML-ready dataset and establish baseline model benchmarks.

Sections:
1. Load & Rebuild Features
2. Encode Categorical Variables
3. Handle Missing Values (ML Strategy)
4. Train / Test Split
5. Baseline Models
6. Feature Importance (Tree-based)
7. Export ML-ready Dataset

## 1 · Load & Rebuild Features

In [ ]:
import glob
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

BASE_DIR   = os.path.dirname(os.getcwd())
SILVER_DIR = os.path.join(BASE_DIR, "data", "silver")
files = sorted(glob.glob(os.path.join(SILVER_DIR, "avito_clean_*.csv")))
assert files, f"No silver files in {SILVER_DIR}"

df = pd.read_csv(files[-1])

# ── Drop known bad rows ───────────────────────────────────────────────────────
BAD_VILLES = {"COURS ET FORMATIONS", "Cours Et Formations"}
df = df[~df["ville"].isin(BAD_VILLES)].reset_index(drop=True)

# ── Drop rows with no prix (target) ──────────────────────────────────────────
df = df[df["prix"].notna()].reset_index(drop=True)

print(f"Rows available for modelling: {len(df)}")

In [ ]:
# ── Rebuild engineered features (same logic as notebook 03) ──────────────────

# Log-transform target
df["log_prix"] = np.log1p(df["prix"])

# Log-transform surface
if "surface_m2" in df.columns:
    df["log_surface"] = np.log1p(df["surface_m2"])

# Total rooms
if "nb_chambres" in df.columns and "nb_salles_bain" in df.columns:
    df["total_rooms"] = df["nb_chambres"].fillna(0) + df["nb_salles_bain"].fillna(0)

# Text-derived flags (only if titre column exists)
if "titre" in df.columns:
    df["is_meuble"] = df["titre"].str.lower().str.contains(
        "meublé|meuble|meubl", na=False).astype(int)
    df["is_bureau"] = df["titre"].str.lower().str.contains(
        "bureau|local|plateau|commercial|professionnel", na=False).astype(int)

# is_grande_ville as int
if "is_grande_ville" in df.columns:
    df["is_grande_ville"] = df["is_grande_ville"].astype(int)

# has_surface flag
if "surface_m2" in df.columns:
    df["has_surface"] = df["surface_m2"].notna().astype(int)

print("Features rebuilt:")
print([c for c in df.columns if c not in ["id", "titre", "lien", "scraped_at", "loaded_at"]])

## 2 · Encode Categorical Variables

In [ ]:
# ── Ordinal encoding: categorie_prix ─────────────────────────────────────────
CATEGORIE_ORDER = {"Très Bas": 0, "Bas": 1, "Moyen": 2, "Élevé": 3, "Luxe": 4, "Inconnu": -1}

if "categorie_prix" in df.columns:
    df["categorie_prix_enc"] = df["categorie_prix"].map(CATEGORIE_ORDER)
    print(f"categorie_prix_enc: {df['categorie_prix_enc'].value_counts().to_dict()}")

# ── Ordinal encoding: prix_type ───────────────────────────────────────────────
PRIX_TYPE_ORDER = {"journalier": 0, "mensuel": 1, "inconnu": -1}

if "prix_type" in df.columns:
    df["prix_type_enc"] = df["prix_type"].map(PRIX_TYPE_ORDER)
    print(f"prix_type_enc: {df['prix_type_enc'].value_counts().to_dict()}")

# ── Target encoding: ville → median log_prix per ville ────────────────────────
# NOTE: In production, compute this only on train set to avoid leakage.
#       With our small dataset (31 rows) we use global median as a proxy.
if "ville" in df.columns:
    ville_median = df.groupby("ville")["log_prix"].transform("median")
    df["ville_target_enc"] = ville_median
    print(f"\nville_target_enc sample:\n{df[['ville','ville_target_enc']].drop_duplicates().sort_values('ville_target_enc', ascending=False).to_string(index=False)}")

## 3 · Handle Missing Values (ML Strategy)

In [ ]:
# ── Define final feature set ──────────────────────────────────────────────────
# We use only features that are:
#   (a) numeric after encoding, and
#   (b) have enough non-null values (≥ 50% filled)

CANDIDATE_FEATURES = [
    "log_surface",
    "surface_m2",
    "nb_chambres",
    "nb_salles_bain",
    "total_rooms",
    "is_meuble",
    "is_bureau",
    "is_grande_ville",
    "has_surface",
    "ville_target_enc",
    "categorie_prix_enc",
    "prix_type_enc",
]
CANDIDATE_FEATURES = [f for f in CANDIDATE_FEATURES if f in df.columns]

TARGET_COL = "log_prix"

# ── Imputation strategy per feature ──────────────────────────────────────────
# Median imputation for numeric features with missing values.
# We do NOT use mean because prix & surface are right-skewed.
imputation_log = []
for feat in CANDIDATE_FEATURES:
    null_count = int(df[feat].isna().sum())
    if null_count > 0:
        median_val = df[feat].median()
        df[feat] = df[feat].fillna(median_val)
        imputation_log.append((feat, null_count, round(median_val, 3)))

if imputation_log:
    print("Median imputation applied:")
    for feat, n, med in imputation_log:
        print(f"  {feat:<25}: filled {n} nulls with median = {med}")
else:
    print("✅ No nulls to impute in selected features")

# ── Final ML dataset ──────────────────────────────────────────────────────────
df_ml = df[CANDIDATE_FEATURES + [TARGET_COL]].dropna().copy()
print(f"\nML dataset shape: {df_ml.shape}")
print(f"Features: {CANDIDATE_FEATURES}")
print(f"Target  : {TARGET_COL}")

## 4 · Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_ml[CANDIDATE_FEATURES]
y = df_ml[TARGET_COL]

# ── IMPORTANT NOTE ────────────────────────────────────────────────────────────
# With only ~28 rows after cleaning, an 80/20 split gives ≈ 5 test rows.
# This is statistically insufficient — results should be treated as
# directional indicators ONLY, not reliable performance estimates.
# Cross-validation would be more appropriate but still unreliable at this scale.
# The real fix is to collect more data (target: ≥ 500 rows).

if len(df_ml) < 20:
    print("⚠️  Dataset too small for a meaningful train/test split.")
    print("    Using leave-one-out cross-validation instead.")
    USE_LOO = True
else:
    USE_LOO = False
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

# ── Scale features ────────────────────────────────────────────────────────────
# Required for linear models; tree-based models don't need it
# but we scale anyway for fair comparison.
scaler = StandardScaler()
if not USE_LOO:
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)
    print("\n✅ Features scaled with StandardScaler")

## 5 · Baseline Models

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

MODELS = {
    "Dummy (median)"        : DummyRegressor(strategy="median"),
    "Ridge Regression"      : Ridge(alpha=1.0),
    "Random Forest"         : RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting"     : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

cv_strategy = LeaveOneOut() if USE_LOO else 5
cv_label    = "LOO CV" if USE_LOO else "5-fold CV"

results = []
X_sc = scaler.fit_transform(X) if USE_LOO else None

print(f"Cross-validation strategy: {cv_label}\n")
for name, model in MODELS.items():
    X_input = X_sc if USE_LOO else X_train_sc
    y_input = y    if USE_LOO else y_train

    # R² and MAE via cross-validation
    r2_scores = cross_val_score(
        model, X_sc if USE_LOO else X, y,
        cv=cv_strategy, scoring="r2"
    )
    mae_scores = cross_val_score(
        model, X_sc if USE_LOO else X, y,
        cv=cv_strategy, scoring="neg_mean_absolute_error"
    )

    mean_r2  = r2_scores.mean()
    mean_mae = -mae_scores.mean()

    results.append({
        "Model"    : name,
        "CV R²"    : round(mean_r2, 3),
        "CV MAE (log)": round(mean_mae, 4),
        "MAE (DH)" : round(np.expm1(mean_mae) , 0),
    })
    print(f"  {name:<25}: R² = {mean_r2:+.3f}  |  MAE (log) = {mean_mae:.4f}")

results_df = pd.DataFrame(results).sort_values("CV R²", ascending=False)
print("\nRanking:")
print(results_df.to_string(index=False))

In [ ]:
# ── Visual: model comparison ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in results_df["CV R²"]]

axes[0].barh(results_df["Model"], results_df["CV R²"], color=colors, edgecolor="white")
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_title(f"CV R² ({cv_label})", fontweight="bold")
axes[0].set_xlabel("R²")
sns.despine(ax=axes[0])

axes[1].barh(results_df["Model"], results_df["CV MAE (log)"], color="#4C72B0", edgecolor="white")
axes[1].set_title(f"CV MAE on log_prix ({cv_label})", fontweight="bold")
axes[1].set_xlabel("MAE (log scale)")
sns.despine(ax=axes[1])

plt.suptitle("Baseline Model Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]["Model"]
print(f"\n🏆 Best model: {best_model_name}")

## 6 · Feature Importance (Tree-based)

In [ ]:
# ── Train Random Forest on full data, extract importances ────────────────────
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=CANDIDATE_FEATURES)
importances = importances.sort_values(ascending=False)

print("Random Forest Feature Importances:\n")
for feat, imp in importances.items():
    bar = "█" * int(imp * 50)
    print(f"  {feat:<25}: {imp:.4f}  {bar}")

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind="barh", ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("Feature Importances — Random Forest", fontsize=13, fontweight="bold")
ax.set_xlabel("Importance")
ax.invert_yaxis()
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## 7 · Export ML-ready Dataset

In [ ]:
import datetime

GOLD_DIR = os.path.join(BASE_DIR, "data", "gold")
os.makedirs(GOLD_DIR, exist_ok=True)

timestamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(GOLD_DIR, f"avito_ml_ready_{timestamp}.csv")

# Export: features + original prix + log_prix
export_cols = CANDIDATE_FEATURES + ["log_prix", "prix"]
export_cols = [c for c in export_cols if c in df_ml.columns]
df_ml[export_cols].to_csv(output_path, index=False)

print(f"✅ ML-ready dataset exported to:")
print(f"   {output_path}")
print(f"   Shape: {df_ml[export_cols].shape}")
print(f"   Columns: {export_cols}")

print("\n⚠️  CRITICAL REMINDER:")
print("   Current dataset has ~28 usable rows.")
print("   All model metrics are directional only.")
print("   Target ≥ 500 rows before drawing conclusions.")